# Model B on Kaggle: latent flow matching + DINOv2 reference cross-attention

In [ ]:
# ---- What this session does ----
RUN_CACHE       = True    # encode train/val sketches+targets with the SD VAE (skipped if a cache is attached)
RUN_VAE_CEILING = True    # encode->decode the test ground truth: the best a latent model can do
RUN_SMOKE       = True    # short run on 1,000 images to check the pipeline end to end
RUN_TRAIN       = False   # full training, resumes from the newest attached checkpoint
RUN_SAMPLE      = False   # test outputs for self_ref / cross_ref / shuffled_ref (default steps and w)
RUN_STEPS_SWEEP = False   # RQ2: N in {1,2,4,8,16,32,50}, self_ref
RUN_W_SWEEP     = False   # RQ1: CFG weight w sweep, self_ref
RUN_EVAL        = False   # score every run present in outputs/

# ---- Settings ----
REPO_URL  = "https://github.com/hidrochin/IT5413-_Anime-line-art-colorization_Project.git"
BRANCH    = "feat/ncthanh-flow"     # the branch with src/models/flow + scripts/train_flow.py
SESSION_MINUTES = 690               # Kaggle stops at 720 min; keep a margin for the final save
SMOKE_STEPS = 3000
STEPS_SWEEP = [1, 2, 4, 8, 16, 32, 50]
W_SWEEP     = [1.0, 1.5, 2.0, 3.0, 4.0, 6.0]

## 0. Setup: code, data, previous outputs

In [ ]:
import glob, os, time
T0 = time.time()

!git clone -q -b {BRANCH} {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!git log -1 --oneline
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Unpack the data to /kaggle/temp (not part of the output).
TAR = glob.glob("/kaggle/input/**/AnimeFaceColorization.tar", recursive=True)[0]
!mkdir -p /kaggle/temp && tar -xf {TAR} -C /kaggle/temp
os.environ["ANIME_DATA_ROOT"] = "/kaggle/temp/AnimeFaceColorization"
os.environ["ANIME_SPLITS_DIR"] = os.path.join(os.path.dirname(TAR), "splits")
print("data:", os.environ["ANIME_DATA_ROOT"], "| splits:", os.environ["ANIME_SPLITS_DIR"])
print(f"setup took {(time.time() - T0) / 60:.1f} min")

In [ ]:
# Model B's own files live in /kaggle/working (= the version's output); test outputs and metric CSVs use
# the repo defaults outputs/ and results/ (also under /kaggle/working/repo).
W = "/kaggle/working"
LATENTS = f"{W}/latents"
RUN_DIR = f"{W}/flow"            # checkpoints/, flow_ema.pt, log.csv, samples/
SMOKE_DIR = f"{W}/flow_smoke"

def newest(pattern):
    found = sorted(glob.glob(pattern, recursive=True))
    return found[-1] if found else None

# Attached previous version of this notebook (if any).
PREV_LATENTS = newest("/kaggle/input/**/latents/train.pt")
PREV_CKPT = newest("/kaggle/input/**/flow/checkpoints/step_*.pt")
PREV_CKPT_DIR = os.path.dirname(PREV_CKPT) if PREV_CKPT else None
print("attached latent cache:", PREV_LATENTS)
print("attached checkpoint:  ", PREV_CKPT)

if PREV_LATENTS and not os.path.exists(f"{LATENTS}/train.pt"):
    # Copy, so this version's output holds the cache too (the next session then needs only one input).
    PREV_LATENTS_DIR = os.path.dirname(PREV_LATENTS)
    !mkdir -p {LATENTS} && cp {PREV_LATENTS_DIR}/*.pt {LATENTS}/

def export_eval_assets():
    if not os.path.exists("data/eval/gt"):
        !python -m scripts.export_eval_assets --out data/eval

## 1. Latent cache (once)

SD-VAE latents (32×32×4, fp16) of every train/val sketch and target, about 1.6 GB for train, so training never
runs the VAE. References are not cached: they are re-warped every step.

In [ ]:
if RUN_CACHE and not os.path.exists(f"{LATENTS}/train.pt"):
    !python -m scripts.cache_latents --splits train val --out {LATENTS} --batch-size 64 --num-workers 4
!ls -lh {LATENTS}

## 2. VAE reconstruction ceiling

Encode→decode the test ground truth. Model B cannot beat these PSNR/SSIM/LPIPS numbers. `evaluate.py` needs
the exported test PNGs (`data/eval`), written once here.

In [ ]:
if RUN_VAE_CEILING:
    export_eval_assets()
    !python -m scripts.vae_ceiling --split test
    !python -m scripts.evaluate --model vae_ceiling --mode self_ref
    !cat results/vae_ceiling_self_ref.csv

## 3. Smoke run (1,000 training images)

Checks data → latents → U-Net → sampling end to end and measures speed (`it_s` in the log). Grid columns:
sketch | reference | seed A | seed B | ground truth. After ~3k steps the colors should roughly follow the
reference; pure noise means something is wrong.

In [ ]:
if RUN_SMOKE:
    !python -m scripts.train_flow --config configs/flow.yaml --latents {LATENTS} --out {SMOKE_DIR} --limit 1000 --max-steps {SMOKE_STEPS}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

def show_run(run_dir):
    log_path = f"{run_dir}/log.csv"
    if not os.path.exists(log_path):
        print("no log at", log_path); return
    log = pd.read_csv(log_path)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(log["step"], log["loss"].rolling(10, min_periods=1).mean(), label="train (smoothed)")
    val = log.dropna(subset=["val_loss"])
    ax[0].plot(val["step"], val["val_loss"], "o-", label="val (EMA)")
    ax[0].set_xlabel("step"); ax[0].set_ylabel("flow-matching loss"); ax[0].legend()
    ax[1].plot(log["step"], log["it_s"]); ax[1].set_xlabel("step"); ax[1].set_ylabel("it/s")
    plt.tight_layout(); plt.show()
    print(log.tail(3).to_string(index=False))
    speed = log["it_s"].tail(20).mean()
    print(f"speed {speed:.2f} it/s -> {speed * 3600:.0f} steps per GPU-hour, {speed * 3600 * 50:.0f} steps in 50 h")
    grid = newest(f"{run_dir}/samples/step_*.png")
    if grid:
        print(grid); display(Image(grid))

if RUN_SMOKE:
    show_run(SMOKE_DIR)

## 4. Full training (resumable)

Resumes from the newest checkpoint in this session's `flow/checkpoints`, else from the attached previous
version (`--init-from`). Saves every 30 min and at the end, and exports the EMA weights to `flow/flow_ema.pt`.
Set `train.max_steps` in `configs/flow.yaml` from the steps per GPU-hour printed by the smoke run (budget ≈ 50 GPU-h).

Slow? If the GPU is idle while the CPU is at 100% (data loading + TPS warp), the CPU is the bottleneck; if the
GPU is saturated, switch to the smaller `arch.block_out_channels` in the config (before the first full session).

In [ ]:
if RUN_TRAIN:
    remaining = int(SESSION_MINUTES - (time.time() - T0) / 60)
    init = f"--init-from {PREV_CKPT_DIR}" if PREV_CKPT_DIR else ""
    print(f"training for up to {remaining} min {init}")
    !python -m scripts.train_flow --config configs/flow.yaml --latents {LATENTS} --out {RUN_DIR} --max-minutes {remaining} {init}
    show_run(RUN_DIR)

## 5. Test outputs (shared `scripts/sample.py`)

Uses the trained EMA weights; if training ended in an earlier version, the attached `flow_ema.pt` is used.
Writes `outputs/flow/<mode>/<id>_s<seed>.png` (5 seeds) and a row in `results/speed.csv`. Rerunning skips
images that already exist.

In [ ]:
CKPT = f"{RUN_DIR}/flow_ema.pt"
if not os.path.exists(CKPT):
    CKPT = newest("/kaggle/input/**/flow/flow_ema.pt") or CKPT
print("weights:", CKPT)
SAMPLE = f"python -m scripts.sample --config configs/flow.yaml --set checkpoint={CKPT} --batch-size 32"

if RUN_SAMPLE:
    export_eval_assets()
    !{SAMPLE} --mode self_ref cross_ref shuffled_ref

## 6. RQ2: quality vs. number of steps

Runs `flow_steps1 … flow_steps50` on self_ref, for `plot_quality_vs_steps.py --series "flow_steps*"`.

In [ ]:
if RUN_STEPS_SWEEP:
    for n in STEPS_SWEEP:
        !{SAMPLE} --mode self_ref --steps {n}

## 7. RQ1: diversity vs. fidelity (CFG weight w)

Runs `flow_w<w>` on self_ref, for `plot_diversity_fidelity.py --series "flow_w*"`. `w = 1.0` is also the
"no guidance" ablation.

In [ ]:
if RUN_W_SWEEP:
    for w in W_SWEEP:
        !{SAMPLE} --mode self_ref --model flow_w{w} --set guidance_scale={w}

## 8. Evaluate every run in `outputs/`

In [ ]:
if RUN_EVAL:
    export_eval_assets()
    for run in sorted(os.listdir("outputs")):
        for mode in sorted(os.listdir(f"outputs/{run}")):
            if not os.path.exists(f"results/{run}_{mode}.csv"):
                !python -m scripts.evaluate --model {run} --mode {mode}
    !ls results

## 9. Before the session ends

Kept in the output: `latents/`, `flow/` (checkpoints, `flow_ema.pt`, `log.csv`, `samples/`), `repo/outputs/`,
`repo/results/`. Copy `results/*.csv`, `flow/log.csv` and a few sample grids into the repo's `results/` and commit
them; never commit checkpoints or generated images.

In [ ]:
!du -sh {W}/* {W}/repo/outputs {W}/repo/results 2>/dev/null
print(f"session time used: {(time.time() - T0) / 60:.0f} min")